In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

In [ ]:
df = pd.read_csv("../data/raw/hdf_kwh_30min.csv")
df.head()

In [ ]:
df = df[["Read Value", "Read Type", "Read Date and End Time"]]
df["end_interval"] = pd.to_datetime(df["Read Date and End Time"], dayfirst = True)
df.head()

In [ ]:
df = df.iloc[::-1]
imp = df[df["Read Type"] == "Active Import Interval (kWh)"].set_index("end_interval")
exp = df[df["Read Type"] == "Active Export Interval (kWh)"].set_index("end_interval")
imp.index = imp.index.tz_localize("Europe/Dublin", ambiguous= "infer") 
exp.index = exp.index.tz_localize("Europe/Dublin", ambiguous= "infer")
len(imp), len(exp)

In [ ]:
df2 = pd.concat([imp["Read Value"].rename("import_kwh"), exp["Read Value"].rename("export_kwh")],axis=1).sort_index()
df2.head(), df2.tail()

In [ ]:
first_export = exp.index.min()
df2.loc[df2.index < first_export, "export_kwh"] = 0.0
df2.index = df2.index - pd.Timedelta(minutes=30)
df2.index.name = "start_interval"
df2.info(), df2.index.tz

In [ ]:
full = pd.date_range(df2.index.min(), df2.index.max(), freq="30min")
missing = full.difference(df2.index)
len(missing)


In [ ]:
df2.to_csv("../data/processed/esb_halfhourly.csv")

In [ ]:
daily = df2.resample("D").sum(min_count=1)
ax = daily.plot(figsize=(12, 4), title="Daily import and export (kWh)")
ax.set_ylabel("kWh per day")
plt.show()

In [ ]:
monthly = df2.resample("MS").sum()
monthly.plot(kind="bar", figsize=(12, 4), title="Monthly import and export (kWh)")
plt.show()

In [ ]:
hourly = df2.groupby(df2.index.hour).mean() * 2     # mean kWh per half-hour x2 = average kW
hourly.plot(figsize=(10, 4), title="Average Power by Time (kW)")
plt.xlabel("Time")
plt.show()

In [ ]:
for name, part in [("Winter 2024/25", df2.loc["2024-12":"2025-02"]),
                   ("Winter 2025/26", df2.loc["2025-12":"2026-02"])]:
    (part.groupby(part.index.hour).mean() * 2)["import_kwh"].plot(label=name)
plt.legend(); plt.show()

In [ ]:
night = df2.between_time("00:00", "03:59")        # half-hours starting 00:00 to 03:30, local time
nightly = night["import_kwh"].resample("D").sum(min_count=1)

ax = nightly.plot(figsize=(12, 4), alpha=0.4, label="Daily")
nightly.rolling(7, center=True).mean().plot(ax=ax, label="7-day average")
ax.axvline(pd.Timestamp("2025-05-29", tz="Europe/Dublin"), color="grey", linestyle="--", label="First export")
ax.set_ylabel("kWh between 00:00 and 04:00")
ax.set_title("Overnight import (00:00 to 04:00) per day")
ax.legend()
plt.show()

In [ ]:
windows = {
    "00:00 to 04:00": ("00:00", "04:00"),
    "06:00 to 10:00": ("06:00", "10:00"),
    "18:00 to 22:00": ("18:00", "22:00"),
}

fig, ax = plt.subplots(figsize=(12, 4))
for label, (start, end) in windows.items():
    w = df2.between_time(start, end, inclusive="left")
    daily = w["import_kwh"].resample("D").sum(min_count=1)
    daily.rolling(7, center=True).mean().plot(ax=ax, label=label)

ax.axvline(pd.Timestamp("2025-05-29", tz="Europe/Dublin"), color="grey", linestyle="--", label="First export")
ax.set_ylabel("kWh imported in window")
ax.set_title("Grid import by time of day")
ax.legend()
plt.show()

In [ ]:
df2["import_kwh"].resample("D").sum().groupby(lambda d: d.day_name()).mean()

In [ ]:
pivot = df2["import_kwh"].groupby([df2.index.month, df2.index.hour]).mean().unstack()
plt.imshow(pivot, aspect="auto"); plt.colorbar(label="kWh per half-hour")
plt.xlabel("Hour"); plt.ylabel("Month"); plt.show()

In [ ]:
from home_energy.load_esb import load_hdf
df_load = load_hdf("../data/raw/hdf_kwh_30min.csv")

assert df_load.equals(df2)